In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# QESEM: una Qiskit Function di Qedma
> **Note:** Le Qiskit Functions sono una funzionalità sperimentale disponibile esclusivamente per gli utenti dei piani IBM Quantum&reg; Premium, Flex e On-Prem (tramite IBM Quantum Platform API). Sono in stato di anteprima e soggette a modifiche.

## Panoramica
Sebbene le unità di elaborazione quantistica abbiano fatto grandi progressi negli ultimi anni, gli errori dovuti al rumore e alle imperfezioni dell'hardware esistente rimangono una sfida centrale per gli sviluppatori di algoritmi quantistici. Con l'avvicinarsi di computazioni quantistiche su scala utilitaria che non possono essere verificate classicamente, le soluzioni per eliminare il rumore con precisione garantita diventano sempre più importanti. Per superare questa sfida, Qedma ha sviluppato Quantum Error Suppression and Error Mitigation (QESEM), integrato senza soluzione di continuità su IBM Quantum Platform come [Qiskit Function](/guides/functions).

Con QESEM, puoi eseguire i tuoi circuiti quantistici su QPU rumorose per ottenere risultati altamente accurati e privi di errori, con overhead di tempo QPU estremamente efficienti, vicini ai limiti fondamentali. Per raggiungere questo obiettivo, QESEM sfrutta una serie di metodi proprietari sviluppati da Qedma per la caratterizzazione e la riduzione degli errori. Le tecniche di riduzione degli errori includono l'ottimizzazione dei gate, la transpilazione consapevole del rumore, la soppressione degli errori (ES) e la mitigazione degli errori non distorta (EM). Grazie a questa combinazione di metodi basati sulla caratterizzazione, puoi ottenere risultati affidabili e privi di errori per circuiti quantistici generici ad alto volume, aprendo la strada ad applicazioni altrimenti impossibili da realizzare.

Per una descrizione completa dei componenti sottostanti e una dimostrazione su scala utilitaria, consulta l'articolo [Reliable high-accuracy error mitigation for utility-scale quantum circuits.](https://arxiv.org/abs/2508.10997)
## Descrizione
Puoi utilizzare la funzione QESEM di Qedma per stimare ed eseguire facilmente i tuoi circuiti con soppressione e mitigazione degli errori, ottenendo volumi di circuiti più grandi e precisioni più elevate. Per usare QESEM, devi fornire un circuito quantistico, un insieme di osservabili da misurare, una precisione statistica target per ciascun osservabile e una QPU scelta. Prima di eseguire il circuito alla precisione target, puoi stimare il tempo QPU necessario sulla base di un calcolo analitico che non richiede l'esecuzione del circuito. Una volta soddisfatto della stima del tempo QPU, puoi eseguire il circuito con QESEM.

Quando esegui un circuito, QESEM avvia un protocollo di caratterizzazione del dispositivo personalizzato per il tuo circuito, producendo un modello di rumore affidabile per gli errori che si verificano nel circuito. Sulla base della caratterizzazione, QESEM implementa innanzitutto la transpilazione consapevole del rumore per mappare il circuito di input su un insieme di qubit fisici e gate, minimizzando il rumore che influisce sull'osservabile target. Questi includono i gate nativamente disponibili (CX/CZ sui dispositivi IBM&reg;), oltre a gate aggiuntivi ottimizzati da QESEM, che formano il set di gate esteso di QESEM. QESEM esegue quindi un insieme di circuiti ES ed EM basati sulla caratterizzazione sulla QPU e raccoglie i risultati delle misurazioni. Questi vengono poi post-elaborati classicamente per fornire un valore di aspettativa non distorto e una barra di errore per ciascun osservabile, corrispondente alla precisione richiesta.

![Panoramica di Qedma QESEM](../docs/images/guides/qedma-qesem/overview.svg)
QESEM ha dimostrato di fornire risultati ad alta precisione per un'ampia varietà di applicazioni quantistiche e sui volumi di circuiti più grandi attualmente raggiungibili. QESEM offre le seguenti funzionalità orientate all'utente, dimostrate nella sezione benchmark qui sotto:
-	**Precisione garantita:** QESEM produce stime non distorte per i valori di aspettativa degli osservabili. Il suo metodo EM è dotato di garanzie teoriche che, insieme alla caratterizzazione all'avanguardia di Qedma, assicurano che la mitigazione converga all'output del circuito privo di rumore entro la precisione specificata dall'utente. A differenza di molti metodi EM euristici soggetti a errori sistematici o distorsioni, la precisione garantita di QESEM è essenziale per garantire risultati affidabili in circuiti quantistici e osservabili generici.
-	**Scalabilità a QPU di grandi dimensioni:** il tempo QPU di QESEM dipende dai volumi dei circuiti, ma è altrimenti indipendente dal numero di qubit. Qedma ha dimostrato QESEM sui dispositivi quantistici più grandi disponibili oggi, inclusi IBM Quantum Eagle a 127 qubit e Heron a 133 qubit.
-	**Indipendente dall'applicazione:** QESEM è stato dimostrato su un'ampia varietà di applicazioni, tra cui simulazione hamiltoniana, VQE, QAOA e stima dell'ampiezza. Puoi inserire qualsiasi circuito quantistico e osservabile da misurare, ottenendo risultati accurati e privi di errori. Le uniche limitazioni sono dettate dalle specifiche hardware e dal tempo QPU allocato, che determinano i volumi di circuiti accessibili e le precisioni di output. Al contrario, molte soluzioni di riduzione degli errori sono specifiche per applicazione o coinvolgono euristiche non controllate, rendendole inapplicabili per circuiti e applicazioni quantistiche generiche.
-  **Set di gate esteso:** QESEM supporta i gate ad angolo frazionario e fornisce gate $Rzz(\theta)$ ad angolo frazionario ottimizzati da Qedma sui dispositivi IBM Quantum Eagle. Questo set di gate esteso consente una compilazione più efficiente e sblocca volumi di circuiti superiori fino a un fattore 2 rispetto alla compilazione CX/CZ predefinita.
-	**Osservabili multi-base:** QESEM supporta osservabili di input composti da molte stringhe di Pauli non commutanti, come gli Hamiltoniani generici. La scelta delle basi di misura e l'ottimizzazione dell'allocazione delle risorse QPU (shot e circuiti) vengono quindi eseguite automaticamente da QESEM per minimizzare il tempo QPU richiesto per la precisione richiesta. Questa ottimizzazione, che tiene conto delle fedeltà hardware e dei tassi di esecuzione, ti consente di eseguire circuiti più profondi e ottenere precisioni più elevate.
## Benchmark
QESEM è stato testato su un'ampia varietà di casi d'uso e applicazioni. I seguenti esempi possono aiutarti a valutare quali tipi di carichi di lavoro puoi eseguire con QESEM.

Una figura di merito fondamentale per quantificare la difficoltà sia della mitigazione degli errori sia della simulazione classica per un dato circuito e osservabile è il **volume attivo**: il numero di gate CNOT che influiscono sull'osservabile nel circuito. Il volume attivo dipende dalla profondità e dalla larghezza del circuito, dal peso dell'osservabile e dalla struttura del circuito, che determina il cono di luce dell'osservabile. Per ulteriori dettagli, vedi il talk dell'[IBM Quantum Summit 2024](https://www.youtube.com/watch?v=Hd-IGvuARfE&t=1730s). QESEM offre un valore particolarmente elevato nel regime ad alto volume, fornendo risultati affidabili per circuiti e osservabili generici.

![Volume attivo](../docs/images/guides/qedma-qesem/active_volume.svg)

| Applicazione    | Numero di qubit | Dispositivo | Descrizione del circuito | Precisione | Tempo totale | Utilizzo runtime |
| ---------  | ---------------- | ----- | -------------------------- | -------- | ---------- | ------------- |
| Circuito VQE  | 8              | Eagle (r3) | 21 strati totali, 9 basi di misura, catena 1D                    | 98%      | 35 min      | 14 min         |
| Kicked Ising   | 28              | Eagle (r3) | 3 strati unici x 3 passi, topologia heavy-hex 2D                      | 97%     | 22 min      | 4 min          |
| Kicked Ising   | 28              | Eagle (r3) | 3 strati unici x 8 passi, topologia heavy-hex 2D                     | 97%      | 116 min      | 23 min          |
| Simulazione hamiltoniana di Trotter   | 40  | Eagle (r3)            | 2 strati unici x 10 passi di Trotter, catena 1D                    | 97%      | 3 ore     | 25 min         |
| Simulazione hamiltoniana di Trotter   | 119  | Eagle (r3)           | 3 strati unici x 9 passi di Trotter, topologia heavy-hex 2D                    | 95%      | 6,5 ore     | 45 min         |
| Kicked Ising  | 136             | Heron (r2) | 3 strati unici x 15 passi, topologia heavy-hex 2D                    | 99%      | 52 min             | 9 min           |

La precisione è misurata qui rispetto al valore ideale dell'osservabile: $\frac{\langle O \rangle_{ideal} - \epsilon}{\langle O \rangle_{ideal}}$, dove '$\epsilon$' è la precisione assoluta della mitigazione (impostata dall'input dell'utente) e $\langle O \rangle_{ideal}$ è l'osservabile nel circuito privo di rumore.
"Utilizzo runtime" misura l'utilizzo del benchmark in modalità batch (somma dell'utilizzo dei singoli job), mentre "tempo totale" misura l'utilizzo in modalità sessione (tempo reale dell'esperimento), che include i tempi di elaborazione classica e comunicazione aggiuntivi. QESEM è disponibile per l'esecuzione in entrambe le modalità, in modo che tu possa fare il miglior uso delle risorse disponibili.

I circuiti Kicked Ising a 28 qubit simulano il Quasicristallo a Tempo Discreto studiato da Shinjo et al. (vedi [arXiv 2403.16718](https://arxiv.org/abs/2403.16718) e [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)) su tre loop connessi di ibm_kawasaki. I parametri del circuito utilizzati qui sono $(\theta_x, \theta_z) = (0.9 \pi, 0)$, con uno stato iniziale ferromagnetico $| \psi_0 \rangle = | 0 \rangle ^{\otimes n}$. L'osservabile misurato è il valore assoluto della magnetizzazione $M = |\frac{1}{28} \sum_{i=0}^{27} \langle Z_i \rangle|$. L'esperimento Kicked Ising su scala utilitaria è stato eseguito sui 136 migliori qubit di ibm_fez; questo benchmark in particolare è stato eseguito all'angolo di Clifford $(\theta_x, \theta_z) = (\pi, 0)$, al quale il volume attivo cresce lentamente con la profondità del circuito, il che — insieme alle elevate fedeltà del dispositivo — consente alta precisione con un breve tempo di esecuzione.

I circuiti di simulazione hamiltoniana di Trotter sono relativi a un modello di Ising a Campo Trasverso ad angoli frazionari: $(\theta_{zz}, \theta_x) = (\pi / 4, \pi /8)$ e $(\theta_{zz}, \theta_x) = (\pi / 6, \pi / 8)$ rispettivamente (vedi [Q2B24 Tokyo](https://www.youtube.com/watch?v=tQW6FdLc6zo)). Il circuito su scala utilitaria è stato eseguito sui 119 migliori qubit di ibm_brisbane, mentre l'esperimento a 40 qubit è stato eseguito sulla migliore catena disponibile. La precisione è riportata per la magnetizzazione; risultati ad alta precisione sono stati ottenuti anche per osservabili di peso maggiore.

Il circuito VQE è stato sviluppato insieme a ricercatori del Center for Quantum Technology and Applications presso il Deutsches Elektronen-Synchrotron (DESY). L'osservabile target era un Hamiltoniano composto da un grande numero di stringhe di Pauli non commutanti, sottolineando le prestazioni ottimizzate di QESEM per gli osservabili multi-base. La mitigazione è stata applicata a un ansatz ottimizzato classicamente; sebbene questi risultati siano ancora inediti, risultati della stessa qualità si otterranno per circuiti diversi con proprietà strutturali simili.
## Per iniziare
Autenticati con la tua [chiave API di IBM Quantum Platform](http://quantum.cloud.ibm.com/) e seleziona la Qiskit Function QESEM come segue. (Questo snippet presuppone che tu abbia già [salvato il tuo account](/guides/functions#install-qiskit-functions-catalog-client) nel tuo ambiente locale.)

In [1]:
import qiskit

from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

qesem_function = catalog.load("qedma/qesem")

## Esempio
Per iniziare, prova questo esempio di base per stimare il tempo QPU necessario all'esecuzione di QESEM per un dato `pub`:

In [ ]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [ ]:
circ = qiskit.QuantumCircuit(5)
circ.cx(0, 1)
circ.cx(2, 3)
circ.cx(1, 2)
circ.cx(3, 4)

avg_magnetization = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("Z", [q], 1 / 5) for q in range(5)], num_qubits=5
)
other_observable = qiskit.quantum_info.SparsePauliOp.from_sparse_list(
    [("ZZ", [0, 1], 1.0), ("XZ", [1, 4], 0.5)], num_qubits=5
)

time_estimation_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    options={
        "estimate_time_only": "analytical",
    },
    backend_name=backend_name,  # E.g. "ibm_fez"
)

time_estimate_result = time_estimation_job.result()

Il seguente esempio esegue un job QESEM:

In [ ]:
sample_job = qesem_function.run(
    pubs=[(circ, [avg_magnetization, other_observable])],
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Puoi usare le familiari API di Qiskit Serverless per controllare lo stato del tuo carico di lavoro Qiskit Function o recuperare i risultati:

In [ ]:
print(sample_job.status())
result = sample_job.result()

## Parametri della funzione
| Nome |  Tipo | Descrizione | Obbligatorio | Predefinito |  Esempio |
| -----| ------| ------------| -------- | ------- | -------- |
| `pubs` | [EstimatorPubLike](/guides/primitive-input-output) | Questo è l'input principale. Il `Pub` contiene 2-4 elementi: un circuito, uno o più osservabili, 0 o un singolo insieme di valori di parametro e una precisione facoltativa. Se la precisione non è specificata, verrà utilizzato il valore `default_precision` da `options`. |  Sì |  N/A |  `[(circuit, [obs1,obs2,obs3], parameter_values, 0.03)]`  |
| `backend_name`| string | Nome del backend da utilizzare | No | QESEM selezionerà il dispositivo meno occupato segnalato da IBM | `"ibm_fez"` |
| `instance` | string | Il nome della risorsa cloud dell'istanza da utilizzare in quel formato | No | N/A | `"CRN"` |
| `options` | dictionary | Opzioni di input. Consulta la sezione **Opzioni** per ulteriori dettagli. | No | Consulta la sezione **Opzioni** per i dettagli. | `{ default_precision = 0.03, "max_execution_time" = 3600, "transpilation_level" = 0}` |

### Opzioni
| Opzione | Scelte | Descrizione | Predefinito |
| -----| -----------| -------- | ------- |
| `estimate_time_only` | `"analytical"` / `"empirical"` / None | Se impostato, il job QESEM calcolerà solo la stima del tempo QPU. Consulta la descrizione seguente per ulteriori dettagli. Se impostato su None, il circuito verrà eseguito con QESEM. | None |
| `default_precision` | 0 < float | Si applica ai `pubs` che non hanno precisione. La precisione indica l'errore accettabile sui valori di aspettativa degli osservabili in valore assoluto. In altre parole, il runtime QPU per la mitigazione sarà determinato per fornire valori di output per tutti gli osservabili di interesse che ricadono in un intervallo di confidenza `1σ` della precisione target. Se vengono forniti più osservabili, la mitigazione verrà eseguita finché la precisione target non viene raggiunta per ciascuno degli osservabili di input. | 0.02 |
| `max_execution_time` | 0 < integer < 28.800 (8 ore) | Ti consente di limitare il tempo QPU, specificato in secondi, da utilizzare per l'intero processo QESEM. Consulta i dettagli aggiuntivi qui sotto. | 3.600 (un'ora) |
| `transpilation_level` | 0 / 1 | Vedi la descrizione qui sotto. | 1 |
| `execution_mode` | `"session"` / `"batch"` | Vedi la descrizione seguente. | "batch" |

> **Caution:** La stima del tempo QPU cambia da un backend all'altro. Pertanto, quando esegui la funzione QESEM, assicurati di eseguirla sullo stesso backend selezionato al momento della stima del tempo QPU.

> **Note:** QESEM terminerà la sua esecuzione quando raggiunge la precisione target o quando raggiunge `max_execution_time`, a seconda di quale evento si verifica prima.

- `estimate_time_only` — Questo flag consente di ottenere una stima del tempo QPU necessario per eseguire il circuito con QESEM.
    - Se impostato su `"analytical"`, viene calcolato un limite superiore del tempo QPU senza consumare alcun utilizzo QPU. Questa stima ha una risoluzione di 30 minuti (ad esempio, 30 minuti, 60 minuti, 90 minuti e così via). È tipicamente pessimistica e può essere ottenuta solo per singoli osservabili di Pauli o somme di Pauli senza supporti intersecanti (ad esempio, Z0+Z1). È principalmente utile per confrontare i livelli di complessità dei diversi parametri forniti dall'utente (circuito, precisione e così via).
    - Per ottenere una stima del tempo QPU più accurata, imposta questo flag su `"empirical"`. Sebbene questa opzione richieda l'esecuzione di un piccolo numero di circuiti, fornisce una stima del tempo QPU significativamente più accurata. Questa stima ha una risoluzione di 5 minuti (ad esempio, 20 minuti, 25 minuti, 30 minuti e così via). Puoi scegliere di eseguire la stima del tempo empirico in modalità batch o sessione. Per ulteriori dettagli, consulta la descrizione di `execution_mode`. Ad esempio, in modalità batch, la stima del tempo empirico consumerà meno di 10 minuti di tempo QPU.

- `max_execution_time`: ti consente di limitare il tempo QPU, specificato in secondi, da utilizzare per l'intero processo QESEM. Poiché il tempo QPU finale necessario per raggiungere la precisione target viene determinato dinamicamente durante il job QESEM, questo parametro ti consente di limitare il costo dell'esperimento. Se il tempo QPU determinato dinamicamente è inferiore al tempo allocato dall'utente, questo parametro non influenzerà l'esperimento. Il parametro `max_execution_time` è particolarmente utile nei casi in cui la stima del tempo analitico fornita da QESEM prima dell'avvio del job è troppo pessimistica e l'utente vuole avviare comunque un job di mitigazione. Dopo il raggiungimento del limite di tempo, QESEM smette di inviare nuovi circuiti. I circuiti già inviati continuano a essere eseguiti (quindi il tempo totale potrebbe superare il limite di 30 minuti) e l'utente riceve i risultati elaborati dai circuiti eseguiti fino a quel momento. Se vuoi applicare un limite di tempo QPU inferiore alla stima del tempo analitico, consulta Qedma per ottenere una stima della precisione raggiungibile entro il limite di tempo.

- `transpilation_level`: dopo che un circuito è stato inviato a QESEM, questo prepara automaticamente diverse transpilazioni alternative del circuito e sceglie quella che minimizza il tempo QPU. Ad esempio, le transpilazioni alternative potrebbero utilizzare gate RZZ frazionari ottimizzati da Qedma per ridurre la profondità del circuito. Naturalmente, tutte le transpilazioni sono equivalenti al circuito di input in termini di output ideale. Per esercitare un controllo maggiore sulla transpilazione del circuito, imposta il livello di transpilazione nelle `options`. Mentre `"transpilation_level": 1` corrisponde al comportamento predefinito descritto sopra, `"transpilation_level": 0` include solo le modifiche minime necessarie al circuito originale; ad esempio, la "layerification" — l'organizzazione delle operazioni del circuito in "strati" di gate a due qubit simultanei. Nota che la mappatura automatica dell'hardware sui qubit ad alta fedeltà viene applicata in ogni caso.

| transpilation_level | Descrizione |
|:-:|:--|
| `1` | Transpilazione QESEM predefinita. Prepara diverse transpilazioni alternative e sceglie quella che minimizza il tempo QPU. Le barriere possono essere modificate nella fase di layerification. |
| `0` | Transpilazione minimale: il circuito mitigato si avvicinerà strutturalmente al circuito di input. I circuiti forniti al livello 0 devono corrispondere alla connettività del dispositivo e devono essere specificati in termini dei seguenti gate: CX, Rzz(α) e gate standard a singolo qubit (U, x, sx, rz e così via). Le barriere saranno rispettate nella fase di layerification. |

- `execution_mode` — Puoi scegliere di eseguire il job QESEM in una [sessione IBM](/guides/execution-modes#session-mode) dedicata o su più [batch IBM](/guides/execution-modes#batch-mode):
    -   **Modalità sessione**: le sessioni sono più costose ma producono un tempo di risposta più breve. Una volta avviata la sessione, la QPU è riservata esclusivamente al job QESEM. Il calcolo dell'utilizzo include sia il tempo trascorso sull'esecuzione QPU sia le elaborazioni classiche associate (eseguite da QESEM e IBM). La Qiskit Function QESEM si occupa di creare e chiudere la sessione automaticamente. Per gli utenti con accesso illimitato alle QPU (ad esempio, installazioni on-premise), si raccomanda di utilizzare la modalità sessione per un'esecuzione QESEM più rapida.
    -   **Modalità batch**: in modalità batch, la QPU viene rilasciata durante le elaborazioni classiche, con un conseguente utilizzo QPU inferiore. Poiché i job batch si estendono tipicamente su un periodo più lungo, vi è un rischio maggiore di derive hardware; QESEM incorpora misure per rilevare e compensare le derive, mantenendo l'affidabilità durante le esecuzioni prolungate.

> **Note:** Le operazioni barrier sono tipicamente usate per specificare gli strati di gate a due qubit nei circuiti quantistici. Al livello 0, QESEM preserva gli strati specificati dalle barrier. Al livello 1, gli strati specificati dalle barrier sono considerati come un'alternativa di transpilazione nella minimizzazione del tempo QPU.
### Output
L'output di una circuito function è un [PrimitiveResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult), che contiene due campi:

- Un oggetto [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult). Può essere indicizzato direttamente dal `PrimitiveResult`.

- Metadati a livello di job.

Ogni `PubResult` contiene un campo `data` e un campo `metadata`.

- Il campo `data` contiene almeno un array di valori di aspettativa (`PubResult.data.evs`) e un array di errori standard (`PubResult.data.stds`). Può anche contenere altri dati, a seconda delle opzioni utilizzate.

- Il campo `metadata` contiene i metadati a livello PUB (`PubResult.metadata`).

Il seguente snippet di codice mostra come recuperare la stima del tempo QPU (quando `estimate_time_only` è impostato):

In [ ]:
print(
    f"The estimated QPU time for this PUB is: \n{time_estimate_result[0].metadata}"
)

Il seguente snippet di codice mostra come recuperare i risultati della mitigazione (quando `estimate_time_only` non è impostato) e le metriche di esecuzione. Questi contengono dati essenziali che consentono una comprensione più approfondita di come i diversi parametri influiscono sull'esecuzione di QESEM. Può essere utile anche quando scrivi un articolo basato sulla tua ricerca.

In [ ]:
results = result[0]
print(f"Mitigated expectation values: \n{results.data.evs}")
print(f"Mitigated error-bar: \n{results.data.stds}")
noisy_results = results.metadata["noisy_results"]
print(f"Noisy expectation values: \n{noisy_results.evs}")
print(f"Noisy error-bar: \n{noisy_results.stds}")
print(f"Total QPU time: \n {results.metadata['total_qpu_time']}")
print(
    f"Gates fidelity measured during the experiment: \n {results.metadata['gate_fidelities']}"
)
print(
    f"Total shots / mitigation shots: \n {results.metadata['total_shots']} / {results.metadata['mitigation_shots']}"
)
print("Transpiled circuits:")
for i, circuit in enumerate(results.metadata["transpiled_circs"]):
    print(f"Circuit {i}:")
    print(f"  Circuit: \n {circuit['circuit']}")
    print(f"  Qubit mapping: \n {circuit['qubit_map']}")
    print(f"  Measurement bases: \n {circuit['num_measurement_bases']}")

## Recupero dei messaggi di errore
Se lo stato del tuo carico di lavoro è ERROR, usa job.result() per recuperare il messaggio di errore come segue:

In [ ]:
print(sample_job.result())

PrimitiveResult([PubResult(data=DataBin(), metadata={'time_estimation_sec': 12600})], metadata={})


## Get support

The Qedma support team is here to help! If you encounter any issues or have questions about using the QESEM Qiskit Function, please don't hesitate to reach out. Our knowledgeable and friendly support staff are ready to assist you with any technical concerns or inquiries you may have.

You can email us at support@qedma.com for assistance. Please include as much detail as possible about the issue you're experiencing to help us provide a swift and accurate response. You can also contact your dedicated Qedma POC representative via email or phone.

To help us assist you more efficiently, please provide the following information when you contact us:

- A detailed description of the issue
- The job ID
- Any relevant error messages or codes


We are committed to providing you with prompt and effective support to ensure you have the best possible experience with our Qiskit Function.

We are always looking to improve our product and we welcome your suggestions! If you have ideas on how we can enhance our services or features you'd like to see, please send us your thoughts at support@qedma.com

## Next steps

<Admonition type="tip" title="Recommendations">

- [Request access to Qedma QESEM](https://quantum.cloud.ibm.com/functions?id=qedma-qesem)
- Review [Bauman, N. P., et al. (2025). Coupled Cluster Downfolding Theory in Simulations of Chemical Systems on Quantum Hardware. arXiv preprint arXiv:2507.01199.](https://arxiv.org/pdf/2507.01199)
- Review [Aharonov, D., et al. (2025). Reliable high-accuracy error mitigation for utility-scale quantum circuits. arXiv preprint arXiv:2508.10997.](https://arxiv.org/pdf/2508.10997)


</Admonition>